# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. The dataset provides detailed record-level clinical and pathological information for cancer survivors with second primary colorectal cancer, including molecular biomarkers and anatomical features.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. This section connects to the schema URL and prints the dataset overview for quick context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for dataset loading
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Access dataset metadata fields directly (do not subscript or iterate)
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their unique `@id` identifiers.

The Croissant schema organizes tabular data into `recordSet`s, each with fields (columns) described by unique `@id`s. We will list all record sets available, and then review their fields and IDs.

In [ ]:
# List all record sets as referenced by their '@id'
record_sets = []
for rs in metadata.recordSet:
    print(f"RecordSet @id: {rs['@id']}   name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])

# For demonstration, examine fields/columns for the first record set
if record_sets:
    first_rs_id = record_sets[0]
    record_set_obj = [rs for rs in metadata.recordSet if rs['@id'] == first_rs_id][0]
    if 'field' in record_set_obj:
        print(f"Fields for RecordSet '{first_rs_id}':")
        for fld in record_set_obj['field']:
            print(f"  Field @id: {fld['@id']} name: {fld.get('name', '')} type: {fld.get('dataType', '')}")
    else:
        print('No fields declared for first RecordSet')
else:
    print('No record sets found in metadata')

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame.

All operations reference entities via their `@id` to ensure consistency and accuracy. Below we extract records from each recordSet and show the data schema for one record.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet '{rs_id}' DataFrame columns:")
    print(df.columns.tolist())
    print(df.head(2)) # Show 2 sample records

# For further EDA, select the main tabular data RecordSet
main_record_set_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
This section demonstrates how to process and analyze the tabular dataset:
- Filter records based on numeric field values
- Normalize numeric attributes
- Group data by key features

Entities are referenced strictly via their `@id` fields for clarity.

In [ ]:
# Example numeric field and group field '@id' selection
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Find a numeric field from the recordSet schema (use @id)
    record_set_obj = [rs for rs in metadata.recordSet if rs['@id'] == main_record_set_id][0]
    numeric_field_id = None
    group_field_id = None
    # Identify fields: for demonstration, pick fields named 'Age' and 'Sex', if present
    for fld in record_set_obj['field']:
        if 'age' in fld.get('name', '').lower():
            numeric_field_id = fld['@id']
        if 'sex' in fld.get('name', '').lower():
            group_field_id = fld['@id']
    # Fallback to first numeric field found
    if not numeric_field_id:
        for fld in record_set_obj['field']:
            if fld.get('dataType', '') in ['Integer', 'Float', 'Number']:
                numeric_field_id = fld['@id']
                break

    # Set arbitrary threshold for EDA
    threshold = 60  # Example: filter patients older than 60
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records with '{numeric_field_id}' > {threshold}")
        print(filtered_df.head())
        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print(f"No numeric field '{numeric_field_id}' found in columns: {df.columns.tolist()}")

    # Group filtered data by group_field (if available)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print(f"No group field '{group_field_id}' found for grouping.")
else:
    print("No main tabular record set available for EDA.")

## 5. Visualization
Visualize the distribution of patient ages, and anatomical site frequency for the filtered cohort.

All plotting is performed directly on DataFrame columns referenced by their corresponding field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    # Plot counts by anatomical site (if available)
    anatomical_field_id = None
    for fld in record_set_obj['field']:
        if 'anatomical' in fld.get('name', '').lower():
            anatomical_field_id = fld['@id']
            break

    if anatomical_field_id and anatomical_field_id in df.columns:
        plt.figure(figsize=(6,3))
        df[anatomical_field_id].value_counts().plot(kind='bar')
        plt.ylabel('Count')
        plt.xlabel(anatomical_field_id)
        plt.title(f"Frequency by '{anatomical_field_id}'")
        plt.show()
else:
    print("Visualization unavailble: required fields missing.")

## 6. Conclusion
This notebook demonstrates loading and exploring the FAIR^2 colorectal cancer survivors dataset using `mlcroissant`. Key steps included:
- Loading metadata and records via Croissant schema URL
- Listing record sets and their fields by `@id`
- Extracting all tabular data into DataFrames, and referencing columns by their unique identifier
- Filtering and normalizing numeric attributes (e.g., age), grouping by demographic features, and plotting distributions

This approach ensures fully FAIR-compliant, reproducible, and programmatic interaction with dataset entities. For advanced analytics, repeat similar workflows referencing only `@id` fields for columns and record sets.